
# SEML Assignment II – Group 66

## Early Diabetes Risk Prediction System

This assignment continues the diabetes prediction project developed in
Assignment I using the Pima Indians Diabetes dataset.

In Assignment I, we analysed the dataset, compared different machine-learning
models and selected Logistic Regression as the final model. We also developed
a Streamlit application for entering patient details and displaying the
prediction.

In Assignment II, we refactored the same ML application using appropriate
object-oriented and functional programming principles. The implementation is
organised into separate modules for data ingestion, feature engineering,
model training and inference. Error handling, logging, a REST API, automated
testing, code formatting and linting, model-quality metrics, data-quality
metrics and deployment-readiness considerations have also been added.

## Group Details and Contributions

| BITS ID | Name | Contribution | Completion |
|---|---|---|---:|
| 2025ab05130 | Megha Ashwin Kanu | Data validation and data-quality metrics; research-versus-production comparison; architecture; requirements traceability; integrated review, report preparation and submission. | 100% |
| 2025AA05152 | K Devi | REST API schemas, input validation, API error handling, API evidence, security considerations and shadow-deployment explanation. | 100% |
| 2025AA05161 | Nandini Agarwal | Unit, data-validation, API, integration, model-training and model-inference tests; coverage; formatting and linting support. | 100% |
| 2025aa05210 | Ajay Nath B. | Modular ML code, configuration, logging, error handling, model evaluation, Streamlit application, model artefacts, GitHub integration and README. | 100% |

## Assignment I Baseline

The original research notebook is retained at:

`notebook/Diabetes_Prediction.ipynb`

Logistic Regression achieved 75.32% test accuracy in the Assignment I
experiments and was selected as the baseline model.

## Assignment II Implementation

The Assignment II solution includes:

- Modular data ingestion, feature engineering, training and inference
- Research-code versus production-code comparison
- Error handling and logging
- FastAPI REST API with request and response validation
- Unit, data-validation, training, inference, API and integration tests
- Accuracy and F1 model-quality metrics
- Schema-validity and invalid-value data-quality metrics
- Black, isort and flake8 evidence
- Shadow deployment for production experimentation
- Security considerations for patient-related data


In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()

required_files = [
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "data" / "diabetes.csv",
    PROJECT_ROOT / "model" / "diabetes_model.pkl",
]

if not all(path.exists() for path in required_files):
    raise FileNotFoundError(
        "Run Group66.ipynb from the root of the project repository."
    )

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Repository initialization: PASSED")


Project root: /content/seml_assignment2_final
Repository initialization: PASSED


## 1. Data Validation and Data-Quality Metrics

The production dataset is validated before preprocessing, training or
inference. Structural errors stop execution, while measurable quality issues
are logged and returned as metrics.

In [2]:

from src.data.loader import load_data
from src.data.validator import validate_dataset

dataset = load_data()
quality = validate_dataset(dataset)

print(f"Rows: {quality['row_count']}")
print(f"Columns: {quality['column_count']}")
print(f"Schema validity: {quality['schema_validity_percent']:.2f}%")
print(f"Missing-value rate: {quality['missing_value_rate_percent']:.2f}%")
print(f"Invalid-zero count: {quality['invalid_zero_count']}")
print(f"Invalid-zero rate: {quality['invalid_zero_rate_percent']:.2f}%")


2026-08-13 21:53:59,567 | INFO | loader.py | Loading dataset from /content/seml_assignment2_final/data/diabetes.csv


2026-08-13 21:53:59,575 | INFO | loader.py | Dataset loaded successfully. Shape: (768, 9)


2026-08-13 21:53:59,576 | INFO | validator.py | Starting dataset validation.


2026-08-13 21:53:59,579 | WARNING | validator.py | Dataset contains 652 medically invalid zero values (16.98% of checked diagnostic values).


2026-08-13 21:53:59,580 | INFO | validator.py | Dataset validation completed successfully. Schema validity=100.00%, missing-value rate=0.00%, invalid-zero rate=16.98%.


Rows: 768
Columns: 9
Schema validity: 100.00%
Missing-value rate: 0.00%
Invalid-zero count: 652
Invalid-zero rate: 16.98%


### Result

The dataset contains 768 rows and 9 columns. Schema validity is 100%, the
missing-value rate is 0%, and 652 invalid diagnostic zeros represent 16.98%
of the checked diagnostic values.

## 2. End-to-End Model Inference

The validated sample is transformed using the saved scaler and passed to the
saved Logistic Regression model.

In [3]:

from src.features.preprocessing import DataPreprocessor
from src.models.predict import DiabetesPredictor

# Use the first dataset row for end-to-end inference.
sample = dataset.drop(columns=["Outcome"]).head(1)
scaled_sample = DataPreprocessor().transform(sample)
prediction, probability = DiabetesPredictor().predict(scaled_sample)

print(f"Prediction: {prediction}")
print(f"Probability: {probability:.4f}")


2026-08-13 21:53:59,612 | INFO | preprocessing.py | Loading scaler.


2026-08-13 21:54:00,369 | INFO | preprocessing.py | Scaling input data.


2026-08-13 21:54:00,383 | INFO | predict.py | Loading prediction model.


2026-08-13 21:54:00,526 | INFO | predict.py | Model loaded successfully.


2026-08-13 21:54:00,565 | INFO | predict.py | Prediction completed. Probability=0.7314


Prediction: 1
Probability: 0.7314


### Result

The end-to-end pipeline returns prediction class 1 with probability 0.7314.

## 3. Model-Quality Metrics

The saved model is evaluated using a reproducible stratified test split with
`random_state=42`. The existing model and scaler are not overwritten.

In [4]:

import joblib
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

# Evaluate the existing saved model without overwriting its artefacts.
features = dataset.drop(columns=["Outcome"])
target = dataset["Outcome"]

_, test_features, _, test_target = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target,
)

scaler = joblib.load("model/scaler.pkl")
model = joblib.load("model/diabetes_model.pkl")

scaled_test_features = scaler.transform(test_features)
test_predictions = model.predict(scaled_test_features)

print(f"Accuracy: {accuracy_score(test_target, test_predictions):.4f}")
print(f"Precision: {precision_score(test_target, test_predictions):.4f}")
print(f"Recall: {recall_score(test_target, test_predictions):.4f}")
print(f"F1 Score: {f1_score(test_target, test_predictions):.4f}")


Accuracy: 0.7143
Precision: 0.6087
Recall: 0.5185
F1 Score: 0.5600


### Result

The model achieved Accuracy 0.7143, Precision 0.6087, Recall 0.5185 and
F1 score 0.5600.

## 4. REST API Verification

FastAPI endpoints are verified for availability, prediction, input validation
and safe internal-error handling.

In [5]:

from unittest.mock import patch
from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

valid_request = {
    "Pregnancies": 6,
    "Glucose": 148,
    "BloodPressure": 72,
    "SkinThickness": 35,
    "Insulin": 125,
    "BMI": 33.6,
    "DiabetesPedigreeFunction": 0.627,
    "Age": 50,
}

root_response = client.get("/")
health_response = client.get("/health")
prediction_response = client.post("/predict", json=valid_request)

invalid_request = valid_request.copy()
invalid_request["Glucose"] = 0
invalid_response = client.post("/predict", json=invalid_request)

with patch(
    "api.main.predict_diabetes",
    side_effect=RuntimeError("simulated internal failure"),
):
    error_response = client.post("/predict", json=valid_request)

print(f"GET /: {root_response.status_code}")
print(f"GET /health: {health_response.status_code}")
print(f"POST /predict: {prediction_response.status_code}")
print(f"Invalid request: {invalid_response.status_code}")
print(f"Internal failure: {error_response.status_code}")
print(f"Safe response: {error_response.json()}")


2026-08-13 21:54:00,889 | INFO | preprocessing.py | Loading scaler.


2026-08-13 21:54:00,891 | INFO | predict.py | Loading prediction model.


2026-08-13 21:54:00,892 | INFO | predict.py | Model loaded successfully.


2026-08-13 21:54:00,925 | INFO | main.py | Root endpoint accessed.


2026-08-13 21:54:00,930 | INFO | main.py | Health endpoint accessed.


2026-08-13 21:54:00,934 | INFO | preprocessing.py | Scaling input data.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
2026-08-13 21:54:00,937 | INFO | predict.py | Prediction completed. Probability=0.7014


2026-08-13 21:54:00,937 | INFO | main.py | Prediction successful: 1


2026-08-13 21:54:00,944 | ERROR | main.py | Prediction failed.
Traceback (most recent call last):
  File "/content/seml_assignment2_final/api/main.py", line 33, in predict
    prediction, probability = predict_diabetes(request)
                              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/unittest/mock.py", line 1139, in __call__
    return self._mock_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/unittest/mock.py", line 1143, in _mock_call
    return self._execute_mock_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/unittest/mock.py", line 1198, in _execute_mock_call
    raise effect
RuntimeError: simulated internal failure


GET /: 200
GET /health: 200
POST /predict: 200
Invalid request: 422
Internal failure: 500
Safe response: {'detail': 'Internal server error'}


### Result

The root, health and valid prediction requests return HTTP 200. Invalid input
returns HTTP 422, while a simulated internal failure returns a safe HTTP 500
response without exposing the underlying exception.

## 5. Automated Tests and Coverage

The pytest suite covers API behaviour, data validation, preprocessing,
training, inference and end-to-end integration.

In [6]:

import subprocess
import sys

test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        "--cov=src",
        "--cov=api",
        "--cov-report=term-missing",
    ],
    text=True,
    capture_output=True,
)

print(test_run.stdout)

if test_run.returncode != 0:
    print(test_run.stderr)
    raise RuntimeError("Automated tests failed.")


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/seml_assignment2_final
configfile: pytest.ini
testpaths: tests
plugins: cov-7.1.0, typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collecting ... collected 27 items

tests/test_api.py::test_root_endpoint PASSED                             [  3%]
tests/test_api.py::test_health_endpoint PASSED                           [  7%]
tests/test_api.py::test_predict_valid_request PASSED                     [ 11%]
tests/test_api.py::test_predict_missing_field PASSED                     [ 14%]
tests/test_api.py::test_predict_invalid_type PASSED                      [ 18%]
tests/test_api.py::test_predict_internal_server_error PASSED             [ 22%]
tests/test_data_validation.py::test_valid_dataset_returns_quality_metrics PASSED [ 25%]
tests/test_data_validation.py::test_empty_dataset_raises_excep

### Result

All 27 automated tests passed, with 100% measured coverage across the tested
`src` and `api` modules.

## 6. Formatting and Linting

Black, isort and flake8 verify code formatting, import ordering and coding
standards across `src`, `api` and `tests`. Before-and-after flake8 evidence is
retained under `reports/`.

In [7]:

import subprocess
import sys

commands = {
    "Black": [sys.executable, "-m", "black", "--check", "src", "api", "tests"],
    "isort": [sys.executable, "-m", "isort", "--check-only", "src", "api", "tests"],
    "flake8": [
        sys.executable,
        "-m",
        "flake8",
        "--max-line-length=88",
        "src",
        "api",
        "tests",
    ],
}

for name, command in commands.items():
    result = subprocess.run(command, text=True, capture_output=True)

    if result.returncode != 0:
        print(f"{name}: FAILED")
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"{name} verification failed.")

    print(f"{name}: PASSED")


Black: PASSED


isort: PASSED


flake8: PASSED


### Result

Black, isort and flake8 passed without violations.

## 7. Research Code versus Production Code

The Assignment I notebook represents research code used for exploration,
preprocessing experiments and model comparison. Assignment II separates these
responsibilities into reusable modules with configuration, logging, exception
handling, API access and automated verification.

The detailed comparison is available in
`reports/research_vs_production.md`.

## 8. Architecture and Requirements Traceability

The production flow is:

`Dataset → Loader → Validator → Feature Processing → Training/Inference → FastAPI/Streamlit`

The detailed architecture and mapping of each Assignment II requirement to
its implementation and evidence are available in
`reports/architecture_and_traceability.md`.

## 9. Production Experimentation and Security

Shadow deployment is proposed for production experimentation. A candidate
model receives copies of live requests, but its predictions do not influence
patient-facing decisions until its behaviour has been evaluated.

Security considerations include strict request validation, safe error
responses, restricted access to patient-related information, secure logging,
encrypted communication and avoiding sensitive data in logs.

Further details are available in
`reports/security_and_production_testing.md`.

## 10. Conclusion

Assignment II converts the Assignment I diabetes research implementation into
a modular ML-enabled system. The final solution includes validation, logging,
exception handling, model training and inference, a REST API, automated tests,
quality metrics, code-quality checks and documented production considerations.

The pipeline, API, test suite, coverage, formatting and linting checks all
passed in the final integrated repository.